# 04. Chronos-Bolt: zero-shot прогноз расходов МО

Фундаментальная модель временных рядов Amazon Chronos-Bolt без дообучения.
История расходов до отсечки используется для прогноза на 12 месяцев: медиана и квантили 0,1/0,9.
Отсечки — декабрь 2023 — ноябрь 2024.

Вход: `data/processed/panel_mo_month.parquet`. Результат: `results/chronos_forecasts.parquet`.
Вычисления выполняются на GPU при его наличии, иначе на CPU.

In [ ]:
import time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from chronos import BaseChronosPipeline

MODEL = "amazon/chronos-bolt-base"
CUTOFFS = pd.date_range("2023-12-01", "2024-11-01", freq="MS")
PRED_LEN, QUANTILES, BATCH = 12, [0.1, 0.5, 0.9], 256
device = "cuda" if torch.cuda.is_available() else "cpu"
print("устройство:", device)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUT = ROOT / "results"; OUT.mkdir(exist_ok=True)
panel = pd.read_parquet(ROOT / "data" / "processed" / "panel_mo_month.parquet")
y = panel[panel.date >= "2023-01-01"].pivot(index="mo_id", columns="date", values="y_all")
print(y.shape)

pipe = BaseChronosPipeline.from_pretrained(MODEL, device_map=device,
                                           torch_dtype=torch.bfloat16 if device == "cuda" else torch.float32)

In [ ]:
rows, t0 = [], time.time()
for T in CUTOFFS:
    hist = y.loc[:, y.columns <= T]
    ids, ctx = [], []
    for mo, s in hist.iterrows():
        s = s.dropna()
        if len(s) >= 6:
            ids.append(mo); ctx.append(torch.tensor(s.values, dtype=torch.float32))
    for i in range(0, len(ctx), BATCH):
        q, mean = pipe.predict_quantiles(context=ctx[i:i + BATCH], prediction_length=PRED_LEN,
                                         quantile_levels=QUANTILES)
        q = q.float().cpu().numpy()
        for j, mo in enumerate(ids[i:i + BATCH]):
            ds = pd.date_range(T + pd.DateOffset(months=1), periods=PRED_LEN, freq="MS")
            rows.append(pd.DataFrame({"mo_id": mo, "origin": T, "ds": ds,
                                      "q10": q[j, :, 0], "yhat": q[j, :, 1], "q90": q[j, :, 2]}))
    print(T.date(), f"{time.time() - t0:.0f} c")

fc = pd.concat(rows, ignore_index=True)
fc["model"] = MODEL
fc.to_parquet(OUT / "chronos_forecasts.parquet", index=False)
print(fc.shape, OUT / "chronos_forecasts.parquet")

MAE Chronos на горизонте 1 месяц по доступным фактическим значениям.

In [ ]:
f1 = fc[fc.ds == fc.origin + pd.DateOffset(months=1)]
truth = y.stack().rename("y_true").reset_index().rename(columns={"date": "ds"})
chk = f1.merge(truth, on=["mo_id", "ds"]).dropna()
print("MAE h=1:", round((chk.yhat - chk.y_true).abs().mean()))